# For defining, validating, and expnading scope of reaction templates

In [ ]:
# File I/O
import json
from collections import defaultdict
from pathlib import Path

# Cheminformatics
from rdkit import Chem
from rdkit.Chem import rdChemReactions

# Custom Imports
from polymerist.rdutils.reactions.reactions import AnnotatedReaction

from polymerist.smileslib.functgroups import FN_GROUP_TABLE
from polymerist.polymers.monomers import specification

from src.reactions import rxn_assemblers, rxn_inputs, test_reactants_catalogue

## Inspecting defined reaction assemblers

### Define functional groups (with R-group linkers) as basis for desired reactions

In [ ]:
# display mechanism schemata prior to generating reactions (allows for inspection of derangement IDs)
for mech_name, mech_schema in rxn_assemblers.items():
    print(mech_name, mech_schema.bond_derangement)
    display(mech_schema.reactants)

In [ ]:
rxns : dict[str, AnnotatedReaction] = {}
for rxnname, rxn_assembler in rxn_assemblers.items():
    # assemble reaction
    rxn = rxn_assembler.assemble_rxn(show_steps=True)
    print(rxnname)
    display(rxn)
    
    rxn.rxnname = rxnname # store mechanism name for reference
    rxns[rxnname] = rxn

In [ ]:
for rxnname, rxn in rxns.items():
    print(rxnname)
    display(rxn)

# Testing that reaction actually behave as intended

In [ ]:
# mechanism = 'polycarbonate_nonphosgene'
# mechanism = 'polyurethane_nonisocyanate'
# mechanism = 'polyimide'
mechanism = 'polyamide'

smarts = rxn_smarts[mechanism]
rxn = AnnotatedReaction.from_smarts(smarts)
rxn.Initialize()
num_warnings, num_errors = rxn.Validate()
print(num_warnings, num_errors)
display(rxn)

test_reactants = test_reactants_catalogue[mechanism]
for reactant in test_reactants:
    display(reactant)

In [ ]:
from polymerist.rdutils.reactions import reactors

reactor = reactors.PolymerizationReactor(rxn)
for product in reactor.react(test_reactants):
    display(product)

In [ ]:
for intermeds, frags in reactor.propagate(test_reactants):
    for inter in intermeds:
        display(inter)

    for frag in frags:
        display(frag)
    print('='*50)